# 15-01 像素误差与任务相关表征

对应第 15 章。只依赖 NumPy 和 Matplotlib，CPU 即可。没有训练 DINO-WM 或 JEPA，而是用手工特征隔离背景与位置的关系。

## 1. 同一个目标，两种不同的错误

目标是把白色方块放到左侧。候选 A 位置正确但背景变亮；候选 B 背景相同但位置移到右侧。先肉眼判断，再计算误差。

In [ ]:
import numpy as np  # 生成可控的教学图像
import matplotlib.pyplot as plt  # 并排显示目标和候选
def scene(x, background):  # 根据中心与背景亮度创建图像
    image = np.full((32, 32), background, dtype=float)  # 创建指定亮度的背景
    image[12:20, x-4:x+4] = 1.0  # 放入固定大小的白色方块
    return image  # 返回单通道图像
goal = scene(10, 0.0)  # 指定目标位置与暗背景
correct_position = scene(10, 0.4)  # 保持位置正确但改变背景
wrong_position = scene(24, 0.0)  # 保持背景正确但改变位置
images = [goal, correct_position, wrong_position]  # 按解释顺序收集图像
titles = ["Goal", "A: right position", "B: wrong position"]  # 用简洁标题说明对照关系
fig, axes = plt.subplots(1, 3, figsize=(9, 3))  # 创建三幅图的画布
for ax, image, title in zip(axes, images, titles):  # 依次显示目标和候选
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)  # 固定亮度范围以公平比较背景
    ax.set_title(title)  # 标明图像对应的候选
    ax.axis("off")  # 隐藏与判断无关的刻度
plt.show()  # 展示视觉结果


### 先读图，不先看分数

左图是目标；中图 A 的方块位置正确，只是大面积背景变亮；右图 B 的方块明显偏离目标。任务只关心位置，所以应偏好 A。固定亮度范围很重要，自动拉伸颜色可能把背景变化隐藏起来。

## 2. 比较两种误差究竟在奖励什么

像素均方误差比较所有像素。手工位置特征只取亮度高于 0.8 的区域中心；这依赖已知颜色规则，不代表真实图像总能如此处理。

In [ ]:
def position_feature(image):  # 提取仅用于教学的方块中心
    ys, xs = np.nonzero(image > 0.8)  # 按阈值找到白色方块
    return np.array([xs.mean(), ys.mean()])  # 返回横向与纵向中心
pixel_errors = [np.mean((candidate - goal) ** 2) for candidate in images[1:]]  # 计算整张图的像素误差
position_errors = [np.linalg.norm(position_feature(candidate) - position_feature(goal)) for candidate in images[1:]]  # 计算位置距离
fig, axes = plt.subplots(1, 2, figsize=(8, 3))  # 分开显示不同单位的误差
axes[0].bar(["A", "B"], pixel_errors, color=["#497f99", "#b67d4c"])  # 左图比较像素误差
axes[0].set(ylabel="Pixel MSE", title="All pixels count")  # 标明亮度误差含义
axes[1].bar(["A", "B"], position_errors, color=["#497f99", "#b67d4c"])  # 右图比较中心距离
axes[1].set(ylabel="Center distance (pixels)", title="Position matters")  # 标明位置误差单位
fig.tight_layout()  # 留出标题与坐标标签空间
plt.show()  # 显示不同评价规则给出的排名
assert pixel_errors[0] > pixel_errors[1]  # 核对背景变化支配了本例的像素比较
assert position_errors[0] == 0 and position_errors[1] > 0  # 核对位置特征正确区分了候选


### 两张图为什么给出相反判断

误差越小越好。左图偏好 B，因为大量背景像素没有变化；右图偏好 A，因为它只比较中心位置。不能比较左右柱子的绝对高度：一个是亮度平方误差，一个是像素距离，单位不同。

这不说明像素训练必然失败，只说明评价指标可能与控制目标不一致。真实模型还需要方向、形状和接触等信息。

## 3. 表征更小，就一定更好吗

只提取整张图像的最大亮度，三张图的表示都是 1。它很好预测，却无法判断是否到达目标。这里直接打印比再画图更清楚。

In [ ]:
collapsed = [float(image.max()) for image in images]  # 用单一亮度模拟丢失位置信息的表征
print("目标、候选 A、候选 B 的单值表征：", collapsed)  # 显示不同场景不可区分
print("三个场景的位置特征：", [position_feature(image).tolist() for image in images])  # 与保留位置信息的表示对照
assert len(set(collapsed)) == 1  # 核对单值表征不能区分图像


### 这不是 JEPA 实现，而是它必须避免的问题

不同场景映射成同一数字，预测误差可能为零，但控制信息已消失。真正的表征学习需要通过预测和任务表现共同检验。

练习：把 A 的背景改成 0.9。阈值特征也会失败，因为背景被纳入白色区域。手工特征依赖假设，不能把本例排序推广到任意图像。